In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import joblib

In [3]:
# CONFIGURATION
TARGET = "fraud_bool"
RANDOM_SEED = 42

# LOAD DATA
print("FAIRFRAUD - CATBOOST BASELINE")
print("\nLoading dataset...")
df = pd.read_csv("../data/dataset.csv")
print(f"Dataset shape: {df.shape}")

# temporal data split
train_df = df[df["month"] <= 4].copy()
val_df = df[df["month"] == 5].copy()
test_df = df[df["month"] >= 6].copy()
print(f"\nTrain shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

FAIRFRAUD - CATBOOST BASELINE

Loading dataset...
Dataset shape: (1000000, 32)

Train shape: (675666, 32)
Validation shape: (119323, 32)
Test shape: (205011, 32)


In [4]:
print("\nFraud distribution:")

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    fraud_rate = data[TARGET].mean() * 100

    print(
        f"{name}: "
        f"{data[TARGET].sum():,} fraud cases "
        f"({fraud_rate:.4f}%)"
    )



Fraud distribution:
Train: 6,740 fraud cases (0.9975%)
Validation: 1,411 fraud cases (1.1825%)
Test: 2,878 fraud cases (1.4038%)


In [7]:
# SEPARATE FEATURES AND TARGET
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

# IDENTIFY CATEGORICAL FEATURES
categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
print("CATEGORICAL FEATURES")
print(categorical_cols)

# CATBOOST REQUIRES CATEGORICAL VALUES AS STRINGS
for col in categorical_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# CLASS IMBALANCE CALCULATION
fraud_count = y_train.sum()
non_fraud_count = len(y_train) - fraud_count
imbalance_ratio = non_fraud_count / fraud_count
print("CLASS IMBALANCE")
print(f"Non-Fraud Cases: {non_fraud_count:,}")
print(f"Fraud Cases: {fraud_count:,}")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}:1")


CATEGORICAL FEATURES
['payment_type', 'employment_status', 'housing_status', 'source', 'device_os', 'age_group']
CLASS IMBALANCE
Non-Fraud Cases: 668,926
Fraud Cases: 6,740
Imbalance Ratio: 99.25:1


In [ ]:
# CATBOOST BASELINE MODEL
print("TRAINING CATBOOST BASELINE")
model = CatBoostClassifier(
    iterations=1000,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_SEED,
    verbose=100,
    early_stopping_rounds=100,
    task_type="GPU",
    devices="0"
)
# TRAIN MODEL
model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(X_val, y_val),
    use_best_model=True
)

TRAINING CATBOOST BASELINE


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8465147	best: 0.8465147 (0)	total: 279ms	remaining: 6m 58s
100:	test: 0.8878863	best: 0.8879093 (99)	total: 8.63s	remaining: 1m 59s
200:	test: 0.8915106	best: 0.8916813 (197)	total: 16.7s	remaining: 1m 47s
300:	test: 0.8913292	best: 0.8918728 (276)	total: 24.8s	remaining: 1m 38s
bestTest = 0.8918728232
bestIteration = 276
Shrink model to first 277 iterations.


CatBoostClassifier(class_weights=[1, np.float64(99.24718100890207)], depth=8, devices='0', early_stopping_rounds=100, eval_metric='AUC', iterations=1500, learning_rate=0.03, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [19]:
# VALIDATION PREDICTIONS
print("\nGenerating validation predictions...")
y_val_prob = model.predict_proba(X_val)[:, 1]
y_val_pred = (
    y_val_prob >= 0.5
).astype(int)
# TEST PREDICTIONS
print("Generating test predictions...")
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (
    y_test_prob >= 0.5
).astype(int)


Generating validation predictions...
Generating test predictions...


In [20]:
# EVALUATION FUNCTION
def evaluate_model(
    y_true,
    y_pred,
    y_prob,
    dataset_name
):
    print(f"{dataset_name.upper()} RESULTS")
    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )
    auc_pr = average_precision_score(
        y_true,
        y_prob
    )
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    print(f"\nROC-AUC: {roc_auc:.6f}")
    print(f"AUC-PR: {auc_pr:.6f}")
    print(f"Recall: {recall:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"F1 Score: {f1:.6f}")
    print("\nConfusion Matrix:")
    cm = confusion_matrix(
        y_true,
        y_pred
    )
    print(cm)
    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            digits=4,
            zero_division=0
        )
    )

    return {
        "Dataset": dataset_name,
        "ROC_AUC": roc_auc,
        "AUC_PR": auc_pr,
        "Recall": recall,
        "Precision": precision,
        "F1_Score": f1
    }



In [21]:
# EVALUATE VALIDATION SET
val_results = evaluate_model(
    y_val,
    y_val_pred,
    y_val_prob,
    "Validation"
)
# EVALUATE TEST SET
test_results = evaluate_model(
    y_test,
    y_test_pred,
    y_test_prob,
    "Test"
)

VALIDATION RESULTS

ROC-AUC: 0.891873
AUC-PR: 0.190405
Recall: 0.715096
Precision: 0.065409
F1 Score: 0.119855

Confusion Matrix:
[[103495  14417]
 [   402   1009]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9961    0.8777    0.9332    117912
           1     0.0654    0.7151    0.1199      1411

    accuracy                         0.8758    119323
   macro avg     0.5308    0.7964    0.5265    119323
weighted avg     0.9851    0.8758    0.9236    119323

TEST RESULTS

ROC-AUC: 0.889120
AUC-PR: 0.193495
Recall: 0.724114
Precision: 0.078722
F1 Score: 0.142005

Confusion Matrix:
[[177744  24389]
 [   794   2084]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9956    0.8793    0.9338    202133
           1     0.0787    0.7241    0.1420      2878

    accuracy                         0.8772    205011
   macro avg     0.5371    0.8017    0.5379    205011
weighted avg     0.9827    0.

In [22]:
from sklearn.metrics import roc_curve


def recall_at_fpr(y_true, y_prob, target_fpr):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    valid_indices = np.where(
        fpr <= target_fpr
    )[0]

    if len(valid_indices) == 0:
        return 0

    max_index = valid_indices[-1]

    return tpr[max_index]

recall_fpr_1 = recall_at_fpr(
    y_test,
    y_test_prob,
    0.01
)

recall_fpr_5 = recall_at_fpr(
    y_test,
    y_test_prob,
    0.05
)

print("\nRECALL AT FIXED FPR")

print(
    f"Recall @ 1% FPR: "
    f"{recall_fpr_1:.4f}"
)

print(
    f"Recall @ 5% FPR: "
    f"{recall_fpr_5:.4f}"
)

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_test_pred
).ravel()

fpr = fp / (fp + tn)

print(f"False Positive Rate: {fpr:.4%}")


RECALL AT FIXED FPR
Recall @ 1% FPR: 0.2450
Recall @ 5% FPR: 0.5368
False Positive Rate: 12.0658%
